In [1]:

import os

In [8]:
%pwd

'd:\\projects\\Kidney-Disease-Classification'

In [3]:

os.chdir("../")

In [4]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [10]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories

In [34]:
from pathlib import Path

# params.yaml is in ROOT directory, config.yaml is in config/ folder
CONFIG_FILE_PATH = Path("config/config.yaml")
PARAMS_FILE_PATH = Path("params.yaml")  # Changed! No "config/" prefix

class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    
    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir 
        )

        return data_ingestion_config

In [15]:
import os
import zipfile
import gdown
from cnnClassifier import logger
from cnnClassifier.utils.common import get_size

In [23]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    
    def download_file(self) -> str:
        '''
        Fetch data from the url
        '''
        try: 
            dataset_url = self.config.source_URL
            zip_download_dir = self.config.local_data_file
            os.makedirs("artifacts/data_ingestion", exist_ok=True)
            logger.info(f"Downloading data from {dataset_url} into file {zip_download_dir}")

            # Use the URL directly from config with fuzzy=True
            gdown.download(dataset_url, zip_download_dir, quiet=False, fuzzy=True)

            logger.info(f"Downloaded data from {dataset_url} into file {zip_download_dir}")

        except Exception as e:
            raise e
        
    
    def extract_zip_file(self):
        """
        zip_file_path: str
        Extracts the zip file into the data directory
        Function returns None
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)

In [35]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e


[2025-11-13 16:43:32,074: INFO: common: yaml file: config\config.yaml loaded successfully]
[2025-11-13 16:43:32,097: INFO: common: yaml file: params.yaml loaded successfully]
[2025-11-13 16:43:32,535: INFO: common: created directory at: artifacts]
[2025-11-13 16:43:32,537: INFO: common: created directory at: artifacts/data_ingestion]
[2025-11-13 16:43:32,539: INFO: 1394041732: Downloading data from https://drive.google.com/uc?export=download&id=1oMKFtXjFRiqM90z-bkkZkhY0LmOVsvhb into file artifacts/data_ingestion/data.zip]


Downloading...
From (original): https://drive.google.com/uc?id=1oMKFtXjFRiqM90z-bkkZkhY0LmOVsvhb
From (redirected): https://drive.google.com/uc?id=1oMKFtXjFRiqM90z-bkkZkhY0LmOVsvhb&confirm=t&uuid=16a237f9-1d3f-4fcc-b0e1-a93ef8d7a19e
To: d:\projects\Kidney-Disease-Classification\artifacts\data_ingestion\data.zip
100%|██████████| 1.63G/1.63G [00:46<00:00, 35.2MB/s]

[2025-11-13 16:44:22,408: INFO: 1394041732: Downloaded data from https://drive.google.com/uc?export=download&id=1oMKFtXjFRiqM90z-bkkZkhY0LmOVsvhb into file artifacts/data_ingestion/data.zip]
